# Fine-tuning a Transformer-based Neural Network with PyTorch — Modern Hugging Face Version

This notebook is a modernization of the original lab.

The biggest change is that **`torchtext` has been removed**. `torchtext` development has stopped, so this version uses the modern NLP stack:

- **PyTorch** for tensor computation and model execution
- **Hugging Face `datasets`** for loading datasets
- **Hugging Face `transformers`** for tokenizers, pretrained Transformer models, and training utilities
- **Hugging Face `evaluate`** plus `scikit-learn` for metrics

The goal is still beginner-friendly:

1. Load a text classification dataset.
2. Tokenize text into model-ready tensors.
3. Fine-tune a pretrained Transformer.
4. Compare full fine-tuning vs. freezing most layers.
5. Practice selectively unfreezing layers.

> Runtime note: The default settings use small subsets so the notebook can run on CPU. Increase the subset sizes and epochs when you have a GPU.

## Table of contents

1. Setup
2. What changed from the old `torchtext` version?
3. Load the IMDB dataset
4. Tokenization
5. Data collators and loaders
6. Fine-tune a pretrained Transformer
7. Evaluate and predict
8. Fine-tune only the classifier head
9. Exercise: unfreeze the last Transformer block
10. Optional: AG News transfer-learning experiment
11. Summary

---

# 1. Setup

Run the installation cell in a fresh environment.

This notebook intentionally does **not** install `torchtext`.

In [ ]:
%%time
%pip install -U \
    torch \
    transformers \
    datasets \
    evaluate \
    accelerate \
    scikit-learn \
    pandas \
    numpy \
    matplotlib \
    tqdm

Restart the kernel after installing packages if your notebook environment asks you to.

In [ ]:
import os
import random
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import evaluate
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

---

# 2. What changed from the old `torchtext` version?

The older notebook used a manually assembled pipeline:

```text
raw text
  -> torchtext tokenizer
  -> torchtext vocabulary
  -> GloVe vectors
  -> custom padding/collate function
  -> custom TransformerEncoder
  -> classifier
```

The modern Hugging Face workflow is:

```text
raw text
  -> AutoTokenizer
  -> input_ids + attention_mask
  -> pretrained Transformer model
  -> classifier head
```

The important conceptual shift:

- In the old notebook, you built most of the NLP pipeline yourself.
- In this notebook, you use a pretrained Transformer checkpoint that already knows its tokenizer and input format.

```mermaid
flowchart LR
    A[Raw review text] --> B[AutoTokenizer]
    B --> C[input_ids]
    B --> D[attention_mask]
    C --> E[Pretrained Transformer]
    D --> E
    E --> F[Classifier head]
    F --> G[Positive / Negative]
```

---

# 3. Load the IMDB dataset

The IMDB dataset is a binary sentiment classification dataset:

- `0` = negative review
- `1` = positive review

To keep the notebook fast, we use a small subset by default.

In [ ]:
raw_imdb = load_dataset("imdb")

raw_imdb

In [ ]:
label_names = ["negative", "positive"]
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in id2label.items()}

id2label, label2id

In [ ]:
# CPU-friendly subset sizes.
# Increase these when you have a GPU.
TRAIN_SIZE = 1_000
VALID_SIZE = 500
TEST_SIZE = 500

small_train = (
    raw_imdb["train"]
    .shuffle(seed=SEED)
    .select(range(TRAIN_SIZE + VALID_SIZE))
    .train_test_split(test_size=VALID_SIZE, seed=SEED)
)

train_ds = small_train["train"]
valid_ds = small_train["test"]
test_ds = raw_imdb["test"].shuffle(seed=SEED).select(range(TEST_SIZE))

train_ds[0]

---

# 4. Tokenization

A tokenizer converts text into integer IDs that the model can process.

For example:

```text
"I loved this movie"
   ↓
[101, 1045, 3866, 2023, 3185, 102]
```

Modern Transformer tokenizers usually return at least:

- `input_ids`: token IDs
- `attention_mask`: which tokens are real tokens versus padding

This notebook uses `distilbert-base-uncased` because it is smaller and faster than BERT while still being Transformer-based.

In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

sample = train_ds[0]["text"]
encoded_sample = tokenizer(sample, truncation=True, max_length=256)

print(sample[:500])
print()
print(encoded_sample.keys())
print(encoded_sample["input_ids"][:20])

In [ ]:
MAX_LENGTH = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = train_ds.map(tokenize_batch, batched=True)
tokenized_valid = valid_ds.map(tokenize_batch, batched=True)
tokenized_test = test_ds.map(tokenize_batch, batched=True)

# Trainer expects the target column to be named "labels".
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_valid = tokenized_valid.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

# Remove raw text because the model does not consume it directly.
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_valid = tokenized_valid.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

tokenized_train[0]

---

# 5. Data collators and loaders

In the old notebook, `collate_batch` padded examples manually.

With Hugging Face, `DataCollatorWithPadding` does this for us.

It dynamically pads each batch to the length of the longest sequence in that batch.

```mermaid
flowchart TD
    A[Example 1: 80 tokens] --> D[Batch padded to 180]
    B[Example 2: 180 tokens] --> D
    C[Example 3: 120 tokens] --> D
    D --> E[input_ids tensor]
    D --> F[attention_mask tensor]
```

Dynamic padding saves memory compared with padding every example to the same global maximum length.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

batch = data_collator([tokenized_train[i] for i in range(3)])
{k: v.shape for k, v in batch.items()}

---

# 6. Fine-tune a pretrained Transformer

Fine-tuning means:

1. Start with a model that already learned general language patterns.
2. Replace or initialize a task-specific classification head.
3. Train the model on your smaller task dataset.

Here, the model predicts two classes: negative or positive.

In [ ]:
def build_model(checkpoint: str = MODEL_CHECKPOINT, num_labels: int = 2):
    return AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )

model = build_model()
model.to(device)

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1_score = f1.compute(predictions=predictions, references=labels, average="binary")["f1"]

    return {
        "accuracy": acc,
        "f1": f1_score,
    }

`TrainingArguments` has changed slightly across `transformers` versions. The helper below keeps the notebook compatible with recent versions by detecting whether your installed version expects `eval_strategy` or `evaluation_strategy`.

In [ ]:
def make_training_args(
    output_dir: str,
    *,
    learning_rate: float = 2e-5,
    num_train_epochs: float = 1.0,
    per_device_train_batch_size: int = 8,
    per_device_eval_batch_size: int = 8,
    weight_decay: float = 0.01,
    logging_steps: int = 25,
):
    signature = inspect.signature(TrainingArguments.__init__)
    kwargs = {
        "output_dir": output_dir,
        "learning_rate": learning_rate,
        "num_train_epochs": num_train_epochs,
        "per_device_train_batch_size": per_device_train_batch_size,
        "per_device_eval_batch_size": per_device_eval_batch_size,
        "weight_decay": weight_decay,
        "logging_steps": logging_steps,
        "save_strategy": "no",
        "report_to": "none",
        "seed": SEED,
    }

    if "eval_strategy" in signature.parameters:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"

    return TrainingArguments(**kwargs)

In [ ]:
training_args = make_training_args(
    output_dir="./imdb-distilbert-full-finetune",
    learning_rate=2e-5,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_valid,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

# Newer transformers versions prefer processing_class.
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

trainer.train()

---

# 7. Evaluate and predict

Validation performance tells you how well the model is doing on held-out data during development.

Test performance should be used less often. Think of the test set as the final exam.

In [ ]:
trainer.evaluate(tokenized_test)

In [ ]:
def predict_sentiment(text: str, model=model, tokenizer=tokenizer):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_id = int(torch.argmax(logits, dim=-1).item())
    probabilities = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    return {
        "label": id2label[predicted_id],
        "probabilities": {
            id2label[i]: float(probabilities[i])
            for i in range(len(id2label))
        },
    }

predict_sentiment("The movie was surprisingly thoughtful, funny, and well acted.")

In [ ]:
predict_sentiment("The story was boring and the acting felt flat.")

---

# 8. Fine-tune only the classifier head

Full fine-tuning updates almost every parameter in the model.

Classifier-head-only fine-tuning freezes the Transformer body and trains only the final classification layer.

This is faster and cheaper, but often less accurate.

```mermaid
flowchart LR
    A[Input text] --> B[Frozen Transformer body]
    B --> C[Trainable classifier head]
    C --> D[Prediction]

    style B stroke-dasharray: 5 5
```

In [ ]:
head_only_model = build_model()
head_only_model.to(device)

# Freeze the Transformer body.
for name, param in head_only_model.named_parameters():
    if not name.startswith("classifier"):
        param.requires_grad = False

for name, param in head_only_model.named_parameters():
    if param.requires_grad:
        print("Trainable:", name)

In [ ]:
head_only_args = make_training_args(
    output_dir="./imdb-distilbert-head-only",
    learning_rate=5e-4,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
)

head_only_trainer_kwargs = {
    "model": head_only_model,
    "args": head_only_args,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_valid,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    head_only_trainer_kwargs["processing_class"] = tokenizer
else:
    head_only_trainer_kwargs["tokenizer"] = tokenizer

head_only_trainer = Trainer(**head_only_trainer_kwargs)

head_only_trainer.train()
head_only_trainer.evaluate(tokenized_test)

---

# 9. Exercise: unfreeze the last Transformer block

A useful compromise is to freeze most of the model and unfreeze only:

- the classifier head
- the last Transformer block

This lets the model adapt some high-level language features without updating the whole network.

For DistilBERT, the Transformer blocks live here:

```python
model.distilbert.transformer.layer
```

Try completing the cell below.

In [ ]:
partial_model = build_model()
partial_model.to(device)

# Freeze everything first.
for param in partial_model.parameters():
    param.requires_grad = False

### EXERCISE:
### Unfreeze the classifier head and the last Transformer block.

# 1. Unfreeze classifier head.
# for param in partial_model.classifier.parameters():
#     ...

# 2. Unfreeze last Transformer block.
# for param in partial_model.distilbert.transformer.layer[-1].parameters():
#     ...

trainable = [name for name, param in partial_model.named_parameters() if param.requires_grad]
trainable[:20], len(trainable)

<details>
<summary>Click here for one solution</summary>

```python
for param in partial_model.classifier.parameters():
    param.requires_grad = True

for param in partial_model.distilbert.transformer.layer[-1].parameters():
    param.requires_grad = True
```

</details>

In [ ]:
# Run this after completing the exercise above.

# partial_args = make_training_args(
#     output_dir="./imdb-distilbert-partial",
#     learning_rate=1e-4,
#     num_train_epochs=1,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
# )
#
# partial_trainer_kwargs = {
#     "model": partial_model,
#     "args": partial_args,
#     "train_dataset": tokenized_train,
#     "eval_dataset": tokenized_valid,
#     "data_collator": data_collator,
#     "compute_metrics": compute_metrics,
# }
#
# if "processing_class" in inspect.signature(Trainer.__init__).parameters:
#     partial_trainer_kwargs["processing_class"] = tokenizer
# else:
#     partial_trainer_kwargs["tokenizer"] = tokenizer
#
# partial_trainer = Trainer(**partial_trainer_kwargs)
# partial_trainer.train()
# partial_trainer.evaluate(tokenized_test)

---

# 10. Optional: AG News transfer-learning experiment

The old notebook used a custom model trained on AG News, then adapted it to IMDB.

With Hugging Face, the common modern pattern is simpler:

1. Start from a general pretrained checkpoint such as `distilbert-base-uncased`.
2. Fine-tune it on the task you care about.

Still, you can experiment with task transfer:

```text
base language model
  -> fine-tune on AG News classification
  -> replace classification head
  -> fine-tune on IMDB sentiment
```

This is optional because it takes longer and task transfer is not always helpful. AG News is topic classification, while IMDB is sentiment classification.

In [ ]:
# Optional sketch only. Uncomment to experiment.

# ag_news = load_dataset("ag_news")
# ag_train = ag_news["train"].shuffle(seed=SEED).select(range(1_000))
# ag_valid = ag_news["test"].shuffle(seed=SEED).select(range(500))
#
# ag_id2label = {
#     0: "World",
#     1: "Sports",
#     2: "Business",
#     3: "Sci/Tech",
# }
# ag_label2id = {v: k for k, v in ag_id2label.items()}
#
# def tokenize_ag(batch):
#     return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
#
# ag_train_tok = ag_train.map(tokenize_ag, batched=True).rename_column("label", "labels").remove_columns(["text"])
# ag_valid_tok = ag_valid.map(tokenize_ag, batched=True).rename_column("label", "labels").remove_columns(["text"])
#
# ag_model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_CHECKPOINT,
#     num_labels=4,
#     id2label=ag_id2label,
#     label2id=ag_label2id,
# )
#
# ag_args = make_training_args(
#     output_dir="./ag-news-distilbert",
#     learning_rate=2e-5,
#     num_train_epochs=1,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
# )
#
# ag_trainer_kwargs = {
#     "model": ag_model,
#     "args": ag_args,
#     "train_dataset": ag_train_tok,
#     "eval_dataset": ag_valid_tok,
#     "data_collator": data_collator,
# }
#
# if "processing_class" in inspect.signature(Trainer.__init__).parameters:
#     ag_trainer_kwargs["processing_class"] = tokenizer
# else:
#     ag_trainer_kwargs["tokenizer"] = tokenizer
#
# ag_trainer = Trainer(**ag_trainer_kwargs)
# ag_trainer.train()
#
# # Save AG News-adapted body.
# ag_model.save_pretrained("./ag-news-distilbert/checkpoint")
# tokenizer.save_pretrained("./ag-news-distilbert/checkpoint")

To adapt the optional AG News model to IMDB, reload the saved checkpoint with `ignore_mismatched_sizes=True` so the old 4-class head can be replaced by a new 2-class head.

```python
imdb_from_ag = AutoModelForSequenceClassification.from_pretrained(
    "./ag-news-distilbert/checkpoint",
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
```

Then train it on the IMDB dataset using the same `Trainer` pattern as above.

---

# 11. Summary

You modernized the old PyTorch + `torchtext` notebook into a current Transformer fine-tuning workflow.

Key takeaways:

- `torchtext` is no longer needed.
- The tokenizer and model checkpoint should come from the same pretrained model family.
- `datasets` handles dataset loading and splitting.
- `DataCollatorWithPadding` replaces most manual padding code.
- `Trainer` handles the training loop, evaluation loop, checkpointing, and metrics.
- Full fine-tuning usually adapts best.
- Head-only fine-tuning is faster but often weaker.
- Unfreezing the last Transformer block is a practical middle ground.

## Mental model

```mermaid
flowchart TD
    A[Dataset] --> B[Tokenizer]
    B --> C[Tokenized dataset]
    C --> D[Data collator]
    D --> E[Trainer]
    F[Pretrained model] --> E
    E --> G[Fine-tuned classifier]
    G --> H[Predictions]
```

## Self-check questions

1. Why should the tokenizer and model checkpoint match?
2. What does `attention_mask` tell the model?
3. Why might full fine-tuning outperform classifier-head-only fine-tuning?
4. Why might you freeze most layers when you have little data?
5. What is the difference between validation accuracy and test accuracy?